<a href="https://colab.research.google.com/github/xwang335/Campbell-A/blob/main/PLS_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
cd /content/drive/MyDrive

/content/drive/MyDrive


In [3]:
import pandas as pd
import numpy as np
from sklearn.cross_decomposition import PLSRegression
import gc

In [4]:
import os

p_path     = "/content/drive/MyDrive/preprocess_data.parquet"
LOCAL_PATH = '/content/df_processed.parquet'

# preprocess and saved to disk
if not os.path.exists(LOCAL_PATH):
    df = pd.read_parquet(p_path)

    # preprocess
    id_cols    = ['DATE', 'permno']
    float_cols = [c for c in df.columns if c not in id_cols]
    df[float_cols] = df[float_cols].astype(np.float32)

    # generate dummy variables
    sic_df        = pd.get_dummies(df['sic2'], prefix='sic')
    sic_cols_list = sic_df.columns.tolist()
    df            = pd.concat([df, sic_df], axis=1)

    # saved to disk
    df.to_parquet(LOCAL_PATH, index=False)
    print(f"finished preprocessing, saved to local disk")

# skip preprocess
else:
    df            = pd.read_parquet(LOCAL_PATH)
    sic_cols_list = [c for c in df.columns if c.startswith('sic_')]
    print(f"read from local disk, skip preprocessing")

print(f"df shape: {df.shape}")

finished preprocessing, saved to local disk
df shape: (3712808, 183)


In [ ]:
# print(len(df[df['exret_lead1'].notna()]))

3712808


In [5]:
def generate_920_features(df, char_cols, macro_cols, sic_cols):

    X_char = df[char_cols].to_numpy(dtype=np.float32, copy=False)
    X_macro = df[macro_cols].to_numpy(dtype=np.float32, copy=False)
    X_sic = df[sic_cols].to_numpy(dtype=np.float32, copy=False)

    # generate interaction (N x 752)

    X_inter = (X_char[:, :, np.newaxis] * X_macro[:, np.newaxis, :]).reshape(len(df), -1)

    # features (94) + interaction (752) + industry (74)
    X_920 = np.hstack([X_char, X_inter, X_sic])

    return X_920

In [6]:
macro=['tbl','d/p','e/p','b/m','tms','dfy','ntis','svar']
features=list(df.columns)[2:96]

In [7]:
def calc_oos_r2(actual, predicted):
    actual    = np.array(actual)
    predicted = np.array(predicted)
    denom = np.sum(actual ** 2)
    if denom == 0:
        return np.nan
    return 1 - np.sum((actual - predicted) ** 2) / denom

In [8]:
def select_n_components(X_train, y_train, X_val, y_val,
                        max_components=10, prev_best_n=None):

    max_components = min(max_components, X_train.shape[1], X_train.shape[0] - 1)

    if prev_best_n is None:
        search_range = range(1, max_components + 1)
    else:
        lo = max(1, prev_best_n - 1)
        hi = min(max_components, prev_best_n + 1)
        search_range = range(lo, hi + 1)

    best_r2, best_n = -np.inf, 1
    for n in search_range:
        pls = PLSRegression(n_components=n, scale=False, max_iter=500, tol=1e-2)
        pls.fit(X_train, y_train)
        r2 = calc_oos_r2(y_val, pls.predict(X_val).ravel())
        if r2 > best_r2:
            best_r2, best_n = r2, n

    return best_n

In [9]:
import warnings
import time
import gc

start_test_year = 1987
end_test_year   = 2016
VAL_YEARS       = 12
MAX_COMPONENTS  = 10
all_preds   = []
prev_best_n = None
dates_all   = df['DATE'].values.copy()
y_all       = df['exret_lead1'].to_numpy(dtype=np.float32, copy=False)
del df
gc.collect()


0

In [ ]:
import pyarrow.parquet as pq
from sklearn.preprocessing import StandardScaler

needed_cols = features + macro + sic_cols_list + ['DATE', 'permno', 'exret_lead1', 'mvel1']
needed_cols = list(dict.fromkeys(needed_cols))
for year in range(start_test_year, end_test_year + 1):
    print(f"\n--- cope with {year} year ---")
    t0 = time.time()
    # split data

    train_mask = (dates_all >= pd.Timestamp(1957, 3, 1)) & \
                 (dates_all <= pd.Timestamp(year - 13, 12, 31))
    val_mask   = (dates_all >= pd.Timestamp(year - 12, 1, 1)) & \
                 (dates_all <= pd.Timestamp(year - 1, 12, 31))
    test_mask  = (dates_all >= pd.Timestamp(year, 1, 1)) & \
                 (dates_all <= pd.Timestamp(year, 12, 31))

    df_year  = pq.read_table(LOCAL_PATH, columns=needed_cols).to_pandas()
    df_train = df_year[train_mask].reset_index(drop=True)
    df_val   = df_year[val_mask].reset_index(drop=True)
    df_test  = df_year[test_mask].reset_index(drop=True)
    del df_year
    gc.collect()

    X_train = generate_920_features(df_train, features, macro, sic_cols_list)
    y_train = y_all[train_mask]
    X_val   = generate_920_features(df_val,   features, macro, sic_cols_list)
    y_val   = y_all[val_mask]
    X_test  = generate_920_features(df_test,  features, macro, sic_cols_list)
    del df_train, df_val
    gc.collect()

    t1 = time.time()
    print(f'  feature construction used: {t1-t0:.1f}s  '
          f'| train rows: {len(X_train):,}  val rows: {len(X_val):,}')

    # parameter selection
    scaler_sel = StandardScaler()
    X_train_s  = scaler_sel.fit_transform(X_train)
    X_val_s    = scaler_sel.transform(X_val)

    best_n = select_n_components(
        X_train_s, y_train, X_val_s, y_val,
        max_components=MAX_COMPONENTS,
        prev_best_n=prev_best_n,
    )
    del X_train_s, X_val_s, scaler_sel
    gc.collect()
    print(f"  best_n={best_n}  select parameter used: {time.time()-t1:.1f}s")

    # train
    X_trainval = np.vstack([X_train, X_val])
    y_trainval = np.concatenate([y_train, y_val])
    del X_train, X_val, y_train, y_val
    gc.collect()

    scaler_final = StandardScaler()
    X_trainval_s = scaler_final.fit_transform(X_trainval)
    X_test_s     = scaler_final.transform(X_test)
    del X_trainval, X_test
    gc.collect()

    t2 = time.time()
    final_model = PLSRegression(n_components=best_n, scale=False,
                                max_iter=500, tol=1e-2)
    final_model.fit(X_trainval_s, y_trainval)
    del X_trainval_s, y_trainval
    gc.collect()
    print(f"  train used: {time.time()-t2:.1f}s  |  total: {time.time()-t0:.1f}s")

    # predict
    res = df_test[['DATE', 'permno', 'exret_lead1', 'mvel1']].copy().reset_index(drop=True)
    res['y_pred'] = final_model.predict(X_test_s).ravel()
    del X_test_s, final_model, df_test, scaler_final
    gc.collect()

    all_preds.append(res)
    prev_best_n = best_n

  # report
results = pd.concat(all_preds, ignore_index=True)

r2_all = calc_oos_r2(results['exret_lead1'], results['y_pred'])

top1000 = (
    results
    .sort_values(['DATE', 'mvel1'], ascending=[True, False])
    .groupby('DATE', sort=False).head(1000)
)
r2_top = calc_oos_r2(top1000['exret_lead1'], top1000['y_pred'])

bot1000 = (
    results
    .sort_values(['DATE', 'mvel1'], ascending=[True, True])
    .groupby('DATE', sort=False).head(1000)
)
r2_bot = calc_oos_r2(bot1000['exret_lead1'], bot1000['y_pred'])

print(f"\n{'='*45}")
print(f"  {'Subsample':<25}  {'OOS R2':>10}")
print(f"{'-'*45}")
print(f"  {'All stocks':<25}  {r2_all*100:>+10.4f}%")
print(f"  {'Top 1000 (largest)':<25}  {r2_top*100:>+10.4f}%")
print(f"  {'Bottom 1000 (smallest)':<25}  {r2_bot*100:>+10.4f}%")
print(f"{'='*45}")



--- cope with 1987 year ---
  feature construction used: 5.3s  | train rows: 472,278  val rows: 764,497
  best_n=1  select parameter used: 275.8s
  train used: 16.1s  |  total: 313.4s

--- cope with 1988 year ---
  feature construction used: 5.6s  | train rows: 530,435  val rows: 788,744
  best_n=2  select parameter used: 38.0s
  train used: 29.4s  |  total: 90.0s

--- cope with 1989 year ---
  feature construction used: 5.8s  | train rows: 588,534  val rows: 814,060
  best_n=2  select parameter used: 83.2s
  train used: 27.6s  |  total: 134.8s

--- cope with 1990 year ---
  feature construction used: 6.0s  | train rows: 647,363  val rows: 836,447
  best_n=2  select parameter used: 73.1s
  train used: 31.7s  |  total: 129.9s

--- cope with 1991 year ---
  feature construction used: 6.4s  | train rows: 704,916  val rows: 859,101
  best_n=2  select parameter used: 71.8s
  train used: 35.1s  |  total: 133.4s

--- cope with 1992 year ---
  feature construction used: 6.6s  | train rows: 76

In [ ]:
# import pyarrow.parquet as pq

# needed_cols = features + macro + sic_cols_list + ['DATE', 'permno', 'exret_lead1', 'mvel1']
# needed_cols = list(dict.fromkeys(needed_cols))

# for year in range(start_test_year, end_test_year + 1):
#     print(f"\n--- cope with {year} year ---")
#     t0 = time.time()

#     train_mask = (dates_all >= pd.Timestamp(1957, 3, 1)) & \
#                  (dates_all <= pd.Timestamp(year - 13, 12, 31))
#     val_mask   = (dates_all >= pd.Timestamp(year - 12, 1, 1)) & \
#                  (dates_all <= pd.Timestamp(year - 1, 12, 31))
#     test_mask  = (dates_all >= pd.Timestamp(year, 1, 1)) & \
#                  (dates_all <= pd.Timestamp(year, 12, 31))


#     df_year  = pq.read_table(LOCAL_PATH, columns=needed_cols).to_pandas()
#     df_train = df_year[train_mask].reset_index(drop=True)
#     df_val   = df_year[val_mask].reset_index(drop=True)
#     df_test  = df_year[test_mask].reset_index(drop=True)
#     del df_year
#     gc.collect()


#     X_train = generate_920_features(df_train, features, macro, sic_cols_list)
#     y_train = y_all[train_mask]
#     X_val   = generate_920_features(df_val,   features, macro, sic_cols_list)
#     y_val   = y_all[val_mask]
#     X_test  = generate_920_features(df_test,  features, macro, sic_cols_list)
#     del df_train, df_val   # 特征生成完再释放
#     gc.collect()

#     t1 = time.time()
#     print(f'  feature construction used: {t1-t0:.1f}s  '
#           f'| train rows: {len(X_train):,}  val rows: {len(X_val):,}')


#     best_n = select_n_components(
#         X_train, y_train, X_val, y_val,
#         max_components=MAX_COMPONENTS,
#         prev_best_n=prev_best_n,
#     )
#     print(f"  best_n={best_n}  select parameter used: {time.time()-t1:.1f}s")


#     X_trainval = np.vstack([X_train, X_val])
#     y_trainval = np.concatenate([y_train, y_val])
#     del X_train, X_val, y_train, y_val
#     gc.collect()


#     t2 = time.time()
#     final_model = PLSRegression(n_components=best_n, scale=False,
#                                 max_iter=500, tol=1e-2)
#     final_model.fit(X_trainval, y_trainval)
#     del X_trainval, y_trainval
#     gc.collect()
#     print(f"  train used: {time.time()-t2:.1f}s  |  total: {time.time()-t0:.1f}s")


#     res = df_test[['DATE', 'permno', 'exret_lead1', 'mvel1']].copy().reset_index(drop=True)
#     res['y_pred'] = final_model.predict(X_test).ravel()
#     del X_test, final_model, df_test
#     gc.collect()

#     all_preds.append(res)
#     prev_best_n = best_n


# results = pd.concat(all_preds, ignore_index=True)

# r2_all = calc_oos_r2(results['exret_lead1'], results['y_pred'])

# top1000 = (
#     results
#     .sort_values(['DATE', 'mvel1'], ascending=[True, False])
#     .groupby('DATE', sort=False).head(1000)
# )
# r2_top = calc_oos_r2(top1000['exret_lead1'], top1000['y_pred'])

# bot1000 = (
#     results
#     .sort_values(['DATE', 'mvel1'], ascending=[True, True])
#     .groupby('DATE', sort=False).head(1000)
# )
# r2_bot = calc_oos_r2(bot1000['exret_lead1'], bot1000['y_pred'])

# print(f"\n{'='*45}")
# print(f"  {'Subsample':<25}  {'OOS R2':>10}")
# print(f"{'-'*45}")
# print(f"  {'All stocks':<25}  {r2_all*100:>+10.4f}%")
# print(f"  {'Top 1000 (largest)':<25}  {r2_top*100:>+10.4f}%")
# print(f"  {'Bottom 1000 (smallest)':<25}  {r2_bot*100:>+10.4f}%")
# print(f"{'='*45}")



--- cope with 1987 year ---
  feature construction used: 5.4s  | train rows: 472,278  val rows: 764,497
  best_n=2  select parameter used: 267.3s
  train used: 26.3s  |  total: 300.7s

--- cope with 1988 year ---
  feature construction used: 5.6s  | train rows: 530,435  val rows: 788,744
  best_n=2  select parameter used: 48.9s
  train used: 29.1s  |  total: 85.3s

--- cope with 1989 year ---
  feature construction used: 5.9s  | train rows: 588,534  val rows: 814,060
  best_n=2  select parameter used: 72.9s
  train used: 27.4s  |  total: 108.0s

--- cope with 1990 year ---
  feature construction used: 6.1s  | train rows: 647,363  val rows: 836,447
  best_n=2  select parameter used: 61.2s
  train used: 31.3s  |  total: 100.5s

--- cope with 1991 year ---
  feature construction used: 6.4s  | train rows: 704,916  val rows: 859,101
  best_n=2  select parameter used: 58.9s
  train used: 35.0s  |  total: 102.4s

--- cope with 1992 year ---
  feature construction used: 6.7s  | train rows: 76

In [ ]:
results

,DATE,permno,exret_lead1,mvel1,y_pred
0,1987-01-30,10000,-0.004300,-0.921324,0.007527
1,1987-02-27,10000,-0.389315,-0.948912,0.008093
2,1987-03-31,10000,-0.066900,-0.949348,0.007885
3,1987-04-30,10000,-0.070467,-0.977465,0.007545
4,1987-01-30,10001,-0.078374,-0.671628,0.013458
...,...,...,...,...,...
2476028,2016-08-31,93436,-0.037840,0.947863,-0.002409
2476029,2016-09-30,93436,-0.031078,0.939954,-0.002108
2476030,2016-10-31,93436,-0.042228,0.936080,-0.002451
2476031,2016-11-30,93436,0.127947,0.936023,-0.002816


In [ ]:
export_path = '/content/drive/MyDrive/_pls.parquet'
results.to_parquet(export_path, index=False)
print(f"to Google Drive: {export_path}")

to Google Drive: /content/drive/MyDrive/_pls.parquet
